In [2]:
import importlib
import torch.nn as nn


import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, Image as HFImage

In [3]:
sys.path.append("..")
from src.data.preprocessing import Ai4MarsSegmentation, IMG_SIZE, CLASS_NAMES, IGNORE_INDEX

SEED = 42
BATCH_SIZE = 8
NUM_WORKERS = 0     # keep 0 while debugging; raise later for speed


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Image size:", IMG_SIZE, "| Batch size:", BATCH_SIZE)

Device: mps
Image size: 512 | Batch size: 8


In [4]:
# Load dataset
IMAGE_COLUMNS = ["image", "label_mask", "rover_mask", "range_mask"]

dataset = load_dataset("hassanjbara/AI4MARS")

raw_datasets = {}
for split, data in dataset.items():
    for column in IMAGE_COLUMNS:
        data = data.cast_column(column, HFImage(decode=False))
    raw_datasets[split] = data

for k, v in raw_datasets.items():
    print(k, len(v))

train 18130
test_min1 322
test_min2 322
test_min3 322


In [5]:
# Datasets and loaders from the saved splits

with open("../data/splits.json") as f:
    splits = json.load(f)                   # Reads the file and converts JSON into a Python dictionary.

train_ds = Ai4MarsSegmentation(raw_datasets["train"], splits["train"])
val_ds   = Ai4MarsSegmentation(raw_datasets["train"], splits["val"])
test_ds  = {                                # Dictionary Comprehension
    s: Ai4MarsSegmentation(raw_datasets[s], range(len(raw_datasets[s])))
    for s in ["test_min1", "test_min2", "test_min3"]
}

pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loaders = {
    s: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                  num_workers=NUM_WORKERS, pin_memory=pin)
    for s, ds in test_ds.items()
}

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: { {s: len(d) for s, d in test_ds.items()} }")

Train: 14312 | Val: 1589 | Test: {'test_min1': 322, 'test_min2': 322, 'test_min3': 322}


             Hugging Face Dataset
                      |
                      v
               raw_datasets
                      |
          +-----------+-----------+
          |           |           |
          v           v           v
       Train         Val         Test
          |           |           |
          v           v           v
      train_ds      val_ds      test_ds
          |           |           |
          v           v           v
    train_loader  val_loader  test_loaders
          |           |           |
          +-----------+-----------+
                      |
                      v
                Model Training
                & Evaluation

In [6]:
# Metrics unit test 
from src.evaluation.metrics import ConfusionMatrix, format_report

targets = torch.tensor([0, 0, 1, 1, 2, 3, 255, 255])
preds   = torch.tensor([0, 1, 1, 1, 2, 0,   0,   3])   # last two are ignored

cm = ConfusionMatrix()
cm.update(preds, targets)
res = cm.compute()

assert res["pixels_evaluated"] == 6                      # 255 pixels skipped
assert abs(res["iou"][0] - 1/3) < 1e-9                   # tp=1, gt=2, pred=2 -> 1/3
assert abs(res["iou"][1] - 2/3) < 1e-9                   # tp=2, gt=2, pred=3 -> 2/3
assert abs(res["iou"][2] - 1.0) < 1e-9
assert res["iou"][3] == 0.0                              # class 3 never predicted correctly
assert abs(res["pixel_acc"] - 4/6) < 1e-9
print(format_report(res))
print("Metrics tests passed.")

mIoU: 0.5000 | pixel acc: 0.6667
  Soil      IoU 0.3333 | recall 0.5000
  Bedrock   IoU 0.6667 | recall 1.0000
  Sand      IoU 1.0000 | recall 1.0000
  Big Rock  IoU 0.0000 | recall 0.0000
Metrics tests passed.


              Model Predictions
                      |
                      v
               Ground Truth
                      |
                      v
                ConfusionMatrix
                      |
                      v
                   update()
                      |
                      v
            Accumulate Pixel Counts
                      |
                      v
                  compute()
                      |
          +-----------+-----------+-----------+
          |           |           |           |
          v           v           v           v
         IoU        Recall    Pixel Acc.     mIoU
          |           |           |           |
          +-----------+-----------+-----------+
                      |
                      v
                format_report()
                      |
                      v
              Evaluation Report

In [7]:
# Model tests
import torch.nn as nn
from src.models.unet import UNet, count_parameters

model = UNet(in_channels=3, num_classes=4, base=32).to(device)
print(f"Parameters: {count_parameters(model) / 1e6:.2f} M")

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

model.train()
logits = model(images)
assert logits.shape == (images.shape[0], 4, IMG_SIZE, IMG_SIZE), logits.shape

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
loss = criterion(logits, labels)
loss.backward()

print("Logits:", tuple(logits.shape), "| Loss:", round(loss.item(), 4))
print("Expected initial loss is about ln(4) =", round(float(np.log(4)), 3))
assert torch.isfinite(loss)
print("Model tests passed.")

Parameters: 7.76 M
Logits: (8, 4, 512, 512) | Loss: 1.4792
Expected initial loss is about ln(4) = 1.386
Model tests passed.


             Input Image
                   |
                   v
                  enc1 -------- e1 ------------------+
                   |                                 |
                  Pool                               |
                   |                                 |
                  enc2 -------- e2 --------------+   |
                   |                             |   |
                  Pool                           |   |
                   |                             |   |
                  enc3 -------- e3 ----------+   |   |
                   |                         |   |   |
                  Pool                       |   |   |
                   |                         |   |   |
                  enc4 -------- e4 ------+   |   |   |
                   |                     |   |   |   |
                  Pool                   |   |   |   |
                   |                     |   |   |   |
              Bottleneck                 |   |   |   |
                   |                     |   |   |   |
                  up4                    |   |   |   |
                   |                     |   |   |   |
                Concat <-----------------+   |   |   |
                   |                         |   |   |
                  dec4                       |   |   |
                   |                         |   |   |
                  up3                        |   |   |
                   |                         |   |   |
                Concat <---------------------+   |   |
                   |                             |   |
                  dec3                           |   |
                   |                             |   |
                  up2                            |   |
                   |                             |   |
                Concat <-------------------------+   |
                   |                                 |
                  dec2                               |
                   |                                 |
                  up1                                |
                   |                                 |
                Concat <-----------------------------+
                   |
                  dec1
                   |
                   v
             Segmentation Head
                   |
                   v
             Output (B, 4, H, W)

In [8]:
import time
import torch.nn as nn
from src.models.unet import UNet

def sync():
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

def benchmark(batch_size, size, base=32, steps=3):
    model = UNet(in_channels=3, num_classes=4, base=base).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = torch.randn(batch_size, 3, size, size, device=device)
    y = torch.randint(0, 4, (batch_size, size, size), device=device)

    def step():
        optimizer.zero_grad(set_to_none=True)
        criterion(model(x), y).backward()
        optimizer.step()

    step(); sync()                       # warm-up (not timed)
    start = time.time()
    for _ in range(steps):
        step()
    sync()
    return (time.time() - start) / steps

for size, bs in [(256, 2), (256, 4), (256, 8), (512, 2), (512, 4)]:
    try:
        sec = benchmark(bs, size)
        print(f"size {size} | batch {bs} | {sec:.2f}s/step | {sec / bs:.2f}s per image")
    except Exception as e:
        print(f"size {size} | batch {bs} | failed: {type(e).__name__}")
        break

size 256 | batch 2 | 0.73s/step | 0.37s per image
size 256 | batch 4 | 62.20s/step | 15.55s per image
size 256 | batch 8 | 118.87s/step | 14.86s per image
size 512 | batch 2 | 125.97s/step | 62.98s per image
size 512 | batch 4 | 106.66s/step | 26.67s per image


In [9]:
# Overfit one fixed batch, with a hard time guard
import time
import torch.nn as nn
from src.training.train import train_one_epoch, evaluate
from src.models.unet import UNet

seed_everything(SEED)
fixed_batch = [next(iter(train_loader))]

model = UNet(in_channels=3, num_classes=4, base=32).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

t0 = time.time()
for step in range(1, 31):
    loss, sec = train_one_epoch(model, fixed_batch, optimizer, criterion, device, desc="overfit")
    if step in (1, 5, 10, 15, 20, 25, 30):
        print(f"step {step:>2} | loss {loss:.4f} | {sec:.2f}s/step")
    if sec > 10:
        print("Step took over 10s: stopping. Tell me and we will not continue locally.")
        break

_, res = evaluate(model, fixed_batch, criterion, device)
print(f"\nmIoU on memorized batch: {res['miou']:.4f} | pixel acc: {res['pixel_acc']:.4f}")
print(f"Total time: {time.time() - t0:.0f}s")

overfit:   0%|          | 0/1 [00:00<?, ?it/s]

step  1 | loss 1.4026 | 272.44s/step
Step took over 10s: stopping. Tell me and we will not continue locally.


eval:   0%|          | 0/1 [00:00<?, ?it/s]


mIoU on memorized batch: 0.1207 | pixel acc: 0.4827
Total time: 297s


In [ ]:
# Reuse the warmed-up model, per-step timing
import time

t0 = time.time()
for step in range(2, 21):
    loss, sec = train_one_epoch(model, fixed_batch, optimizer, criterion, device, desc="overfit")
    print(f"step {step:>2} | loss {loss:.4f} | {sec:.2f}s")
    if sec > 10:
        print("Still slow after warm-up: stop here.")
        break

_, res = evaluate(model, fixed_batch, criterion, device)
print(f"\nmIoU on memorized batch: {res['miou']:.4f} | pixel acc: {res['pixel_acc']:.4f}")
print(f"Total time: {time.time() - t0:.0f}s")

overfit:   0%|          | 0/1 [00:00<?, ?it/s]

step  2 | loss 1.1824 | 187.50s
Still slow after warm-up: stop here.


eval:   0%|          | 0/1 [00:00<?, ?it/s]


mIoU on memorized batch: 0.1231 | pixel acc: 0.4220
Total time: 225s


              Model (U-Net)
                    |
                    v
              Training Loop
                    |
                    v
              train_one_epoch()
                    |
          +---------+---------+
          |                   |
          v                   v
      Forward Pass       Calculate Loss
          |                   |
          +---------+---------+
                    |
                    v
              Backpropagation
                    |
                    v
              Update Weights
                    |
                    v
             Validation Loop
                    |
                    v
                evaluate()
                    |
          +---------+---------+
          |                   |
          v                   v
      Validation Loss    ConfusionMatrix
                              |
                              v
                         Compute Metrics
                              |
                              v
                             mIoU
                              |
                              v
                       Save Best Model
                              |
                              v
                       Next Epoch